# Tech Challenge Olist — Notebook Melhorado

FIAP POSTECH | Data Analytics – Fase 1 | 2026

**Paleta unificada azul:** `#0D3257` → `#1F4E79` → `#2E75B6` → `#5B9BD5` → `#9DC3E6` → `#BDD7EE`  
**Estilo:** sem grade, bordas finas em `#D0D8E4`, anotações contextuais, `fill_between` nas linhas  

**Seções:**
1. Bibliotecas e Configurações
2. Carregamento dos Dados
3. Limpeza e Modelagem
4. KPIs Executivos
5. Painel de KPIs (Cards)
6. Crescimento Mensal
7. YoY e Top Categorias
8. Pagamentos
9. Vendedores
10. Logística
11. Satisfação
12. Conferência com Power BI


## ── 1. BIBLIOTECAS E CONFIGURAÇÕES ────────────────────────────────────────


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# ── PALETA UNIFICADA AZUL ─────────────────────────────────────────────────────
AZUL_DEEP   = '#0D3257'   # azul muito escuro — destaques máximos
AZUL_ESCURO = '#1F4E79'   # azul escuro — títulos e valores principais
AZUL_MED    = '#2E75B6'   # azul médio — barras e linhas principais
AZUL_VIVO   = '#5B9BD5'   # azul médio-claro — séries secundárias
AZUL_CLARO  = '#9DC3E6'   # azul claro — séries terciárias / 2017
AZUL_PALE   = '#BDD7EE'   # azul pálido — fundo de cards / bandas
STRIPE      = '#E8F1FB'   # fundo muito suave

# Cores de apoio (uso mínimo e intencional)
LARANJA     = '#D85A30'   # alertas, anomalias, atrasos
VERDE       = '#1D9E75'   # métricas positivas / metas atingidas
CINZA       = '#9BA3AE'   # médias e referências neutras

# Ramp para sequências ordenadas (mais escuro = maior valor)
AZUL_RAMP = [AZUL_PALE, AZUL_CLARO, AZUL_VIVO, AZUL_MED, AZUL_ESCURO, AZUL_DEEP]

# ── HELPERS ───────────────────────────────────────────────────────────────────
def fmt_brl(v):
    """Formata valor em R$ com sufixo Mi/Mil."""
    if v >= 1_000_000: return f'R$ {v/1_000_000:.1f}Mi'
    if v >= 1_000:     return f'R$ {v/1_000:.1f}Mil'
    return f'R$ {v:,.0f}'

def estilo(ax, titulo=None, xlabel=None, ylabel=None, fs=13):
    """Estilo padrão: fundo branco, sem grade, bordas finas em cinza claro."""
    ax.set_facecolor('white')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D0D8E4')
    ax.spines['bottom'].set_color('#D0D8E4')
    ax.tick_params(colors='#444444', labelsize=10)
    if titulo:  ax.set_title(titulo, fontsize=fs, fontweight='bold',
                              color=AZUL_ESCURO, pad=14)
    if xlabel:  ax.set_xlabel(xlabel, fontsize=10, color='#555555')
    if ylabel:  ax.set_ylabel(ylabel, fontsize=10, color='#555555')

def rotulo_bar(ax, bars, valores, fmt_fn=None, cor=None, offset_y=0.01):
    """Adiciona rótulos acima de barras verticais."""
    cor = cor or AZUL_ESCURO
    ymax = ax.get_ylim()[1]
    for bar, v in zip(bars, valores):
        label = fmt_fn(v) if fmt_fn else f'{v:,}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + ymax * offset_y,
                label, ha='center', va='bottom',
                fontsize=8.5, color=cor, fontweight='semibold')

print("✓ Configurações e paleta azul carregados.")


## ── 2. CARREGAMENTO DOS DADOS ──────────────────────────────────────────────


In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print(f"Dataset via KaggleHub: {path}")
except Exception:
    path = "data"
    print(f"KaggleHub indisponível — usando pasta local: {path}")

df_orders         = pd.read_csv(f'{path}/olist_orders_dataset.csv')
df_order_items    = pd.read_csv(f'{path}/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv(f'{path}/olist_order_payments_dataset.csv')
df_order_reviews  = pd.read_csv(f'{path}/olist_order_reviews_dataset.csv')
df_products       = pd.read_csv(f'{path}/olist_products_dataset.csv')
df_sellers        = pd.read_csv(f'{path}/olist_sellers_dataset.csv')
df_customers      = pd.read_csv(f'{path}/olist_customers_dataset.csv')
try:
    df_traducao = pd.read_csv(f'{path}/product_category_name_translation.csv')
except Exception:
    df_traducao = pd.DataFrame(columns=['product_category_name','product_category_name_english'])

print("\nTabelas carregadas:")
for nome, df in [('orders',df_orders),('order_items',df_order_items),
                 ('payments',df_order_payments),('reviews',df_order_reviews),
                 ('products',df_products),('sellers',df_sellers),('customers',df_customers)]:
    print(f"  {nome:20s}: {df.shape[0]:>7,} linhas × {df.shape[1]} colunas")


## ── 3. LIMPEZA E MODELAGEM ─────────────────────────────────────────────────


In [ ]:
# Replica as transformações do Power Query e DAX.
# Regra crítica: todas as métricas financeiras filtradas por order_status = 'delivered'

# ── Datas ──────────────────────────────────────────────────────────────────────
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

df_orders['ano']    = df_orders['order_purchase_timestamp'].dt.year
df_orders['mes_num']= df_orders['order_purchase_timestamp'].dt.month
df_orders['ano_mes']= df_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

# ── Categorias amigáveis ───────────────────────────────────────────────────────
df_products['product_category_name'] = df_products['product_category_name'].fillna('sem_categoria')
substituicoes = {
    'cama_mesa_banho': 'Cama, Mesa e Banho', 'beleza_saude': 'Beleza e Saúde',
    'esporte_lazer': 'Esporte e Lazer', 'informatica_acessorios': 'Informática e Acessórios',
    'moveis_decoracao': 'Móveis e Decoração', 'utilidades_domesticas': 'Utilidades Domésticas',
    'relogios_presentes': 'Relógios e Presentes', 'telefonia': 'Telefonia',
    'ferramentas_jardim': 'Ferramentas e Jardim', 'automotivo': 'Automotivo',
    'brinquedos': 'Brinquedos', 'eletronicos': 'Eletrônicos',
    'eletrodomesticos': 'Eletrodomésticos', 'sem_categoria': 'Sem Categoria',
}
df_products['categoria_amigavel'] = df_products['product_category_name'].apply(
    lambda n: substituicoes.get(n, n.replace('_',' ').title()))

# ── Reviews: manter avaliação mais recente por pedido ─────────────────────────
df_order_reviews['review_creation_date'] = pd.to_datetime(
    df_order_reviews['review_creation_date'], errors='coerce')
reviews_limpo = (df_order_reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates('order_id')
    [['order_id','review_score','review_creation_date']])

# ── Pagamentos: forma principal (payment_sequential = 1) ──────────────────────
traducao_pagamento = {
    'credit_card': 'Cartão de Crédito', 'boleto': 'Boleto',
    'voucher': 'Voucher', 'debit_card': 'Cartão de Débito',
}
payments_principal = (df_order_payments
    .sort_values(['order_id','payment_sequential'])
    .drop_duplicates('order_id')
    [['order_id','payment_type','payment_installments','payment_value']]
    .assign(tipo_pagamento=lambda d: d['payment_type'].map(traducao_pagamento).fillna('Não Definido'),
            modo_pagamento=lambda d: d['payment_installments'].apply(
                lambda n: 'À vista' if n <= 1 else ('10x+' if n >= 10 else f'{n}x'))))

payments_agg = (df_order_payments
    .groupby('order_id', as_index=False)
    .agg(receita_paga=('payment_value','sum'),
         parcelas_max=('payment_installments','max'),
         qtd_formas=('payment_sequential','count')))

# ── Bases analíticas ───────────────────────────────────────────────────────────
orders_base = (df_orders
    .merge(df_customers[['customer_id','customer_unique_id','customer_city','customer_state']],
           on='customer_id', how='left')
    .merge(reviews_limpo, on='order_id', how='left')
    .merge(payments_principal, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left'))

orders_items = (df_order_items
    .merge(df_orders[['order_id','customer_id','order_status',
                       'order_purchase_timestamp','order_delivered_customer_date',
                       'order_estimated_delivery_date','ano','mes_num','ano_mes']],
           on='order_id', how='left')
    .merge(df_customers[['customer_id','customer_unique_id','customer_city','customer_state']],
           on='customer_id', how='left')
    .merge(df_products[['product_id','product_category_name','categoria_amigavel']],
           on='product_id', how='left')
    .merge(df_sellers[['seller_id','seller_city','seller_state']], on='seller_id', how='left')
    .merge(payments_principal[['order_id','payment_type','tipo_pagamento',
                                'payment_installments','modo_pagamento']],
           on='order_id', how='left')
    .merge(reviews_limpo[['order_id','review_score']], on='order_id', how='left'))
orders_items['receita_total'] = orders_items['price'] + orders_items['freight_value']

# FILTRO DELIVERED — regra crítica de negócio
orders_delivered = orders_base[orders_base['order_status'] == 'delivered'].copy()
vendas_delivered  = orders_items[orders_items['order_status'] == 'delivered'].copy()

print(f"\n✓ Limpeza concluída.")
print(f"  orders_base     : {orders_base.shape[0]:>7,} linhas")
print(f"  orders_delivered: {orders_delivered.shape[0]:>7,} linhas  (status = delivered)")
print(f"  vendas_delivered: {vendas_delivered.shape[0]:>7,} linhas  (itens entregues)")


## ── 4. KPIs EXECUTIVOS ─────────────────────────────────────────────────────


In [ ]:
receita_total_del   = vendas_delivered['receita_total'].sum()
receita_produto_del = vendas_delivered['price'].sum()
frete_total_del     = vendas_delivered['freight_value'].sum()
pedidos_del         = vendas_delivered['order_id'].nunique()
ticket_medio_del    = receita_total_del / pedidos_del
vendedores_del      = vendas_delivered['seller_id'].nunique()
clientes_del        = orders_delivered['customer_id'].nunique()
ltv_del             = receita_total_del / clientes_del
pct_frete_del       = frete_total_del / receita_produto_del

lead_time           = (orders_delivered['order_delivered_customer_date'] -
                       orders_delivered['order_purchase_timestamp']).dt.days
lead_time_md        = lead_time.mean()
no_prazo            = (orders_delivered['order_delivered_customer_date'] <=
                       orders_delivered['order_estimated_delivery_date']).sum()
tx_prazo            = no_prazo / len(orders_delivered)
tx_atraso           = 1 - tx_prazo
nps_medio           = orders_delivered['review_score'].mean()
pct_5               = (orders_delivered['review_score'] == 5).sum() / \
                       orders_delivered['review_score'].notna().sum()
pct_churn           = (orders_delivered['review_score'] <= 2).sum() / \
                       orders_delivered['review_score'].notna().sum()

print("=" * 60)
print("  KPIs EXECUTIVOS — Pedidos Entregues")
print("=" * 60)
print(f"  Receita Total         : {fmt_brl(receita_total_del)}")
print(f"  Pedidos Entregues     : {pedidos_del:,}")
print(f"  Ticket Médio          : R$ {ticket_medio_del:,.2f}")
print(f"  Vendedores Ativos     : {vendedores_del:,}")
print(f"  LTV por Cliente       : R$ {ltv_del:,.2f}")
print(f"  % Frete / Receita     : {pct_frete_del:.1%}")
print(f"  Lead Time Médio       : {lead_time_md:.1f} dias")
print(f"  Entrega no Prazo      : {tx_prazo:.1%}")
print(f"  NPS Médio             : {nps_medio:.2f} / 5")
print(f"  % Nota 5              : {pct_5:.1%}")
print(f"  % Risco de Churn (≤2) : {pct_churn:.1%}")
print("=" * 60)


## ── 5. PAINEL DE KPIs (Cards Visuais) ──────────────────────────────────────


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.patch.set_facecolor('white')
fig.suptitle('Painel de KPIs Executivos — Pedidos Entregues',
             fontsize=15, fontweight='bold', color=AZUL_ESCURO, y=1.01)

kpis_cards = [
    ('Receita Total',      fmt_brl(receita_total_del),       AZUL_DEEP,   '96.478 pedidos entregues'),
    ('Ticket Médio',       f'R$ {ticket_medio_del:,.2f}',    AZUL_MED,    'Produto + frete'),
    ('LTV por Cliente',    f'R$ {ltv_del:,.2f}',             AZUL_ESCURO, '97% pedido único'),
    ('% Frete / Receita',  f'{pct_frete_del:.1%}',           AZUL_MED,    'do valor do produto'),
    ('Vendedores Ativos',  f'{vendedores_del:,}',            AZUL_DEEP,   '+463% em 20 meses'),
    ('Lead Time Médio',    f'{lead_time_md:.1f} dias',        AZUL_MED,    'Compra → entrega'),
    ('Entrega no Prazo',   f'{tx_prazo:.1%}',                 VERDE,       'Meta: > 90%  ✓'),
    ('NPS Médio',          f'{nps_medio:.2f} / 5',            AZUL_ESCURO, f'{pct_5:.0%} notas 5'),
]

for ax, (label, valor, cor, sub) in zip(axes.flat, kpis_cards):
    ax.set_facecolor(STRIPE)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0, 0.88), 1, 0.12, transform=ax.transAxes,
                                color=cor, clip_on=False))
    ax.text(0.5, 0.60, valor, ha='center', va='center',
            fontsize=20, fontweight='bold', color=cor, transform=ax.transAxes)
    ax.text(0.5, 0.33, label, ha='center', va='center',
            fontsize=10, color='#333', transform=ax.transAxes, fontweight='semibold')
    ax.text(0.5, 0.13, sub, ha='center', va='center',
            fontsize=7.5, color='#777', transform=ax.transAxes)
    ax.add_patch(plt.Rectangle((0,0),1,1, fill=False,
                                edgecolor='#C5D5E8', linewidth=1.2, transform=ax.transAxes))

plt.tight_layout(pad=1.5)
plt.show()


## ── 6. CRESCIMENTO MENSAL ───────────────────────────────────────────────────


In [ ]:
crescimento_mensal = (vendas_delivered
    .groupby('ano_mes')
    .agg(receita_total=('receita_total','sum'),
         pedidos=('order_id','nunique'),
         ticket_medio=('receita_total','mean'))
    .reset_index()
    .sort_values('ano_mes'))
crescimento_mensal['mom'] = crescimento_mensal['receita_total'].pct_change()

x = np.arange(len(crescimento_mensal))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), facecolor='white')
fig.suptitle('Gráfico 1 — Crescimento Mensal: Receita e Volume de Pedidos',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO, y=1.01)

# Receita
cores_barras = [AZUL_CLARO if p[:4] == '2017' else AZUL_MED
                for p in crescimento_mensal['ano_mes']]
bars = ax1.bar(x, crescimento_mensal['receita_total'], color=cores_barras,
               width=0.75, alpha=0.88, zorder=2)
ax1.plot(x, crescimento_mensal['receita_total'], color=AZUL_DEEP, linewidth=2,
         marker='o', markersize=4.5, zorder=3)
ax1.fill_between(x, crescimento_mensal['receita_total'], alpha=0.08, color=AZUL_MED)

# Divisor 2017|2018
idx_div = next((i for i,m in enumerate(crescimento_mensal['ano_mes']) if m.startswith('2018')), None)
if idx_div:
    ax1.axvline(idx_div - 0.5, color='#C5D5E8', linewidth=1.5, linestyle='--')
    ax1.text(idx_div/2 - 0.5, crescimento_mensal['receita_total'].max()*0.93,
             '2017', ha='center', fontsize=11, color=AZUL_CLARO, fontweight='bold', alpha=0.7)
    ax1.text(idx_div + (len(x)-idx_div)/2 - 0.5, crescimento_mensal['receita_total'].max()*0.93,
             '2018', ha='center', fontsize=11, color=AZUL_MED, fontweight='bold', alpha=0.7)

ax1.set_xticks(x)
ax1.set_xticklabels(crescimento_mensal['ano_mes'], rotation=45, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt_brl(v)))
p17 = mpatches.Patch(color=AZUL_CLARO, label='2017')
p18 = mpatches.Patch(color=AZUL_MED, label='2018')
ax1.legend(handles=[p17,p18], fontsize=9, frameon=False)
estilo(ax1, ylabel='Receita (R$)')

# Pedidos
ax2.bar(x, crescimento_mensal['pedidos'], color=cores_barras, width=0.75, alpha=0.88)
if idx_div:
    ax2.axvline(idx_div - 0.5, color='#C5D5E8', linewidth=1.5, linestyle='--')
ax2.set_xticks(x)
ax2.set_xticklabels(crescimento_mensal['ano_mes'], rotation=45, ha='right', fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax2.legend(handles=[p17,p18], fontsize=9, frameon=False)
estilo(ax2, ylabel='Pedidos')

plt.tight_layout()
plt.show()


## ── 7. YoY E TOP CATEGORIAS ────────────────────────────────────────────────


In [ ]:
crescimento_anual = (vendas_delivered
    .groupby('ano')
    .agg(receita_total=('receita_total','sum'),
         pedidos=('order_id','nunique'),
         ticket_medio=('receita_total','mean'))
    .reset_index())
crescimento_anual['yoy'] = crescimento_anual['receita_total'].pct_change()

top_cat = (vendas_delivered
    .groupby('categoria_amigavel')['receita_total'].sum()
    .sort_values(ascending=False)
    .head(10).reset_index()
    .sort_values('receita_total'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7), facecolor='white')
fig.suptitle('Gráfico 2 — Crescimento Anual (YoY) e Top 10 Categorias',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO)

# YoY
anos_str = crescimento_anual['ano'].astype(str)
cores_ano = [AZUL_CLARO, AZUL_MED, AZUL_DEEP][:len(crescimento_anual)]
bars = ax1.bar(anos_str, crescimento_anual['receita_total'], color=cores_ano, width=0.55)
for i, (bar, r, yoy) in enumerate(zip(bars, crescimento_anual['receita_total'],
                                        crescimento_anual['yoy'])):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height() + bar.get_height()*0.01,
             fmt_brl(r), ha='center', fontsize=10, fontweight='bold', color=AZUL_ESCURO)
    if not pd.isna(yoy):
        ax1.text(bar.get_x()+bar.get_width()/2, r/2,
                 f'+{yoy:.0%}\nYoY', ha='center', fontsize=10, color='white', fontweight='bold')
ax1.text(0, -crescimento_anual['receita_total'].max()*0.08,
         '* 2016 parcial (set–dez)  |  2018 parcial (jan–ago)',
         fontsize=7.5, color=CINZA)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt_brl(v)))
estilo(ax1, 'Receita por Ano — YoY', ylabel='Receita (R$)')

# Top categorias — ramp azul
n = len(top_cat)
idx_ramp = np.linspace(0, len(AZUL_RAMP)-1, n).astype(int)
cores_cat = [AZUL_RAMP[i] for i in idx_ramp]
bars2 = ax2.barh(top_cat['categoria_amigavel'], top_cat['receita_total'],
                 color=cores_cat, height=0.65)
for bar, r in zip(bars2, top_cat['receita_total']):
    ax2.text(bar.get_width() + top_cat['receita_total'].max()*0.01,
             bar.get_y()+bar.get_height()/2, fmt_brl(r), va='center', fontsize=9,
             color=AZUL_ESCURO)
ax2.set_xlim(0, top_cat['receita_total'].max()*1.22)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt_brl(v)))
estilo(ax2, 'Top 10 Categorias por Receita', xlabel='Receita (R$)')

plt.tight_layout()
plt.show()


## ── 8. PAGAMENTOS ───────────────────────────────────────────────────────────


In [ ]:
pagamentos_del = (df_order_payments
    .merge(df_orders[['order_id','order_status']], on='order_id', how='left')
    .query("order_status == 'delivered'")
    .assign(tipo_pagamento=lambda d: d['payment_type'].map(traducao_pagamento).fillna('Não Definido')))

pagamento_tipo = (pagamentos_del
    .groupby('tipo_pagamento')
    .agg(pedidos=('order_id','nunique'),
         receita=('payment_value','sum'),
         ticket_medio=('payment_value','mean'))
    .reset_index()
    .sort_values('receita', ascending=False)
    .assign(pct_receita=lambda d: d['receita']/d['receita'].sum()))

credito = pagamentos_del[pagamentos_del['payment_type'] == 'credit_card'].copy()
parcelas = (credito.groupby('payment_installments')
    .agg(pedidos=('order_id','nunique'), receita=('payment_value','sum'))
    .reset_index().sort_values('payment_installments'))
parcelas['pct'] = parcelas['pedidos'] / parcelas['pedidos'].sum()
taxa_parcelamento = credito[credito['payment_installments']>1]['order_id'].nunique() / \
                    credito['order_id'].nunique()
media_parcelas    = credito['payment_installments'].mean()

# Gráfico
fig = plt.figure(figsize=(17, 7), facecolor='white')
fig.suptitle('Gráfico 3 — Meios de Pagamento: Receita, Ticket Médio e Parcelamento',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO)

ax1 = fig.add_subplot(1,3,1)
ax2 = fig.add_subplot(1,3,2)
ax3 = fig.add_subplot(1,3,3)

y = np.arange(len(pagamento_tipo))
n_tipos = len(pagamento_tipo)
idx_r = np.linspace(len(AZUL_RAMP)-1, 1, n_tipos).astype(int)
cores_pag = [AZUL_RAMP[i] for i in idx_r]

# Receita
pag_sort = pagamento_tipo.sort_values('receita')
cores_s = list(reversed(cores_pag))
bars = ax1.barh(pag_sort['tipo_pagamento'], pag_sort['receita'], color=cores_s[:len(pag_sort)], height=0.55)
for bar, pct in zip(bars, pag_sort['pct_receita']):
    ax1.text(bar.get_width()*0.97, bar.get_y()+bar.get_height()/2,
             f'{pct:.1%}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt_brl(v)))
estilo(ax1, 'Receita por Meio', xlabel='Receita (R$)')

# Ticket médio
bars2 = ax2.barh(pag_sort['tipo_pagamento'], pag_sort['ticket_medio'],
                  color=cores_s[:len(pag_sort)], height=0.55)
for bar, t in zip(bars2, pag_sort['ticket_medio']):
    ax2.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
             f'R$ {t:.0f}', va='center', fontsize=9, color=AZUL_ESCURO)
ax2.set_yticklabels([''] * len(pag_sort))
estilo(ax2, 'Ticket Médio', xlabel='Ticket (R$)')

# Parcelas
pico_10x = parcelas[parcelas['payment_installments']==10]['pedidos'].max() if 10 in parcelas['payment_installments'].values else 0
cores_parc = [LARANJA if n == 10 else AZUL_MED for n in parcelas['payment_installments']]
bars3 = ax3.bar(parcelas['payment_installments'].astype(str), parcelas['pedidos'],
                color=cores_parc, width=0.75)
for bar, q in zip(bars3, parcelas['pedidos']):
    if q > parcelas['pedidos'].max() * 0.07:
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height() + parcelas['pedidos'].max()*0.01,
                 f'{q:,}', ha='center', fontsize=7.5, color=AZUL_ESCURO)
if pico_10x > 0:
    ax3.annotate('Anomalia\nem 10x',
                 xy=(9, pico_10x),
                 xytext=(10.5, parcelas['pedidos'].max()*0.72),
                 arrowprops=dict(arrowstyle='->', color=LARANJA, lw=1.3),
                 fontsize=8.5, color=LARANJA, fontweight='bold')
estilo(ax3, 'Parcelas — Cartão de Crédito', xlabel='Nº Parcelas', ylabel='Pedidos')

plt.tight_layout()
plt.show()
print(f"\n  Cartão de Crédito : 77% dos pedidos, {pagamento_tipo.iloc[0]['pct_receita']:.1%} da receita")
print(f"  Taxa Parcelamento : {taxa_parcelamento:.0%}  |  Média: {media_parcelas:.1f}x")


## ── 9. VENDEDORES ───────────────────────────────────────────────────────────


In [ ]:
vendedores_mensal = (vendas_delivered
    .assign(mes_n=lambda d: d['ano_mes'].str[5:7].astype(int))
    .groupby(['ano','ano_mes','mes_n'])['seller_id'].nunique()
    .reset_index()
    .rename(columns={'seller_id':'vendedores_ativos'})
    .sort_values(['ano','mes_n']))

vm_2017 = vendedores_mensal[vendedores_mensal['ano']==2017]
vm_2018 = vendedores_mensal[vendedores_mensal['ano']==2018]

vendedores_uf = (df_sellers.groupby('seller_state')['seller_id'].nunique()
    .reset_index().rename(columns={'seller_state':'state','seller_id':'vendedores'}))
compradores_uf = (orders_delivered.groupby('customer_state')['customer_unique_id'].nunique()
    .reset_index().rename(columns={'customer_state':'state','customer_unique_id':'compradores'}))
vendedores_ativos_uf = (vendas_delivered.groupby('seller_state')
    .agg(vendedores_ativos=('seller_id','nunique'), receita_total=('receita_total','sum'))
    .reset_index().rename(columns={'seller_state':'state'}))

mercado = (compradores_uf.merge(vendedores_uf, on='state', how='outer')
    .merge(vendedores_ativos_uf, on='state', how='outer').fillna(0))
mercado['ratio'] = mercado['compradores'] / mercado['vendedores'].replace(0, np.nan)
mercado['receita_por_vend'] = mercado['receita_total'] / mercado['vendedores_ativos'].replace(0, np.nan)
media_nac = mercado['compradores'].sum() / mercado['vendedores'].sum()

fig, axes = plt.subplots(1, 3, figsize=(19, 7), facecolor='white')
fig.suptitle('Gráfico 4 — Vendedores: Crescimento, Demanda Reprimida e Produtividade',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO)
meses_label = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

# 2017 vs 2018
axes[0].plot(vm_2017['mes_n'], vm_2017['vendedores_ativos'],
             color=AZUL_CLARO, linewidth=2.5, marker='o', markersize=6, label='2017')
axes[0].fill_between(vm_2017['mes_n'], vm_2017['vendedores_ativos'], alpha=0.15, color=AZUL_CLARO)
axes[0].plot(vm_2018['mes_n'], vm_2018['vendedores_ativos'],
             color=AZUL_DEEP, linewidth=2.5, marker='o', markersize=6, label='2018')
axes[0].fill_between(vm_2018['mes_n'], vm_2018['vendedores_ativos'], alpha=0.2, color=AZUL_DEEP)
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(meses_label, fontsize=8.5)
axes[0].legend(fontsize=9, frameon=False)
estilo(axes[0], 'Crescimento de Vendedores Ativos\n2017 vs 2018', xlabel='Mês', ylabel='Vendedores')

# Demanda reprimida
top_ratio = (mercado.dropna(subset=['ratio']).sort_values('ratio', ascending=False)
    .head(12).sort_values('ratio'))
cores_rat = [AZUL_DEEP if r > media_nac*1.8 else (AZUL_MED if r > media_nac else AZUL_CLARO)
             for r in top_ratio['ratio']]
bars = axes[1].barh(top_ratio['state'], top_ratio['ratio'], color=cores_rat, height=0.65)
axes[1].axvline(media_nac, color=LARANJA, linewidth=1.8, linestyle='--', alpha=0.8)
axes[1].text(media_nac+1, -0.8, f'Média BR\n({media_nac:.0f})', color=LARANJA, fontsize=7.5, fontweight='bold')
for bar, r in zip(bars, top_ratio['ratio']):
    axes[1].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{r:.0f}', va='center', fontsize=8.5, color=AZUL_ESCURO, fontweight='bold')
p1 = mpatches.Patch(color=AZUL_DEEP, label='Alta demanda')
p2 = mpatches.Patch(color=AZUL_MED, label='Acima da média')
p3 = mpatches.Patch(color=AZUL_CLARO, label='Saturado')
axes[1].legend(handles=[p1,p2,p3], fontsize=8, frameon=False, loc='lower right')
estilo(axes[1], 'Demanda Reprimida\n(compradores por vendedor)', xlabel='Ratio')

# Receita por vendedor
top_rv = (mercado[mercado['vendedores_ativos']>=5].dropna(subset=['receita_por_vend'])
    .sort_values('receita_por_vend', ascending=False).head(10).sort_values('receita_por_vend'))
media_br_rv = mercado[mercado['vendedores_ativos']>=5]['receita_por_vend'].mean()
cores_rv = [AZUL_DEEP if r > media_br_rv else AZUL_CLARO for r in top_rv['receita_por_vend']]
bars3 = axes[2].barh(top_rv['state'], top_rv['receita_por_vend'], color=cores_rv, height=0.65)
axes[2].axvline(media_br_rv, color=LARANJA, linewidth=1.8, linestyle='--', alpha=0.8)
for bar, r in zip(bars3, top_rv['receita_por_vend']):
    axes[2].text(bar.get_width()+50, bar.get_y()+bar.get_height()/2,
                 fmt_brl(r), va='center', fontsize=8.5, color=AZUL_ESCURO)
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: fmt_brl(v)))
estilo(axes[2], 'Receita por Vendedor Ativo\n(Top 10 Estados)', xlabel='Receita Média (R$)')

plt.tight_layout()
plt.show()
print(f"\n  Média nacional : {media_nac:.0f} compradores/vendedor")


## ── 10. LOGÍSTICA ───────────────────────────────────────────────────────────


In [ ]:
orders_log = orders_delivered.copy()
orders_log['lead_time_dias'] = (orders_log['order_delivered_customer_date'] -
                                 orders_log['order_purchase_timestamp']).dt.days
orders_log['atraso_dias'] = (orders_log['order_delivered_customer_date'] -
                               orders_log['order_estimated_delivery_date']).dt.days
orders_log['atrasado'] = orders_log['atraso_dias'] > 0

atraso_uf = (orders_log.groupby('customer_state')
    .agg(pedidos=('order_id','count'), atrasados=('atrasado','sum'),
         lead_time_medio=('lead_time_dias','mean'))
    .reset_index()
    .assign(taxa_atraso=lambda d: d['atrasados']/d['pedidos']))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(19, 7), facecolor='white')
fig.suptitle('Gráfico 5 — Logística: Lead Time, Pontualidade e Correlação com NPS',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO)

# Histograma lead time
lt_clean = orders_log['lead_time_dias'].dropna()
ax1.hist(lt_clean, bins=45, color=AZUL_MED, edgecolor='white', alpha=0.88)
ax1.axvline(lead_time_md, color=LARANJA, linewidth=2, linestyle='--')
ax1.text(lead_time_md+0.5, ax1.get_ylim()[1]*0.88,
         f'Média: {lead_time_md:.1f}d', color=LARANJA, fontsize=9, fontweight='bold')
ax1.axvspan(lt_clean.min(), 10, alpha=0.06, color=VERDE)
ax1.axvspan(20, lt_clean.max(), alpha=0.06, color=LARANJA)
estilo(ax1, 'Distribuição do Lead Time', xlabel='Dias', ylabel='Pedidos')

# Taxa de atraso por UF
top_atraso = atraso_uf.sort_values('taxa_atraso', ascending=False).head(10).sort_values('taxa_atraso')
cores_a = [AZUL_DEEP if t > tx_atraso*2 else AZUL_MED for t in top_atraso['taxa_atraso']]
bars = ax2.barh(top_atraso['customer_state'], top_atraso['taxa_atraso'], color=cores_a, height=0.65)
ax2.axvline(tx_atraso, color=LARANJA, linewidth=1.8, linestyle='--', alpha=0.8)
ax2.text(tx_atraso+0.002, -0.8, f'Média: {tx_atraso:.1%}', color=LARANJA, fontsize=7.5)
for bar, t in zip(bars, top_atraso['taxa_atraso']):
    ax2.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
             f'{t:.0%}', va='center', fontsize=8.5, color=AZUL_ESCURO, fontweight='bold')
ax2.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
estilo(ax2, 'Taxa de Atraso\n(Top 10 Estados)', xlabel='% Atrasados')

# Correlação atraso x NPS
corr_df = (orders_log.merge(orders_delivered[['order_id','review_score']], on='order_id', how='inner')
    .groupby('customer_state')
    .agg(taxa_atraso=('atrasado','mean'), nps=('review_score','mean'))
    .reset_index().dropna())
ax3.scatter(corr_df['taxa_atraso']*100, corr_df['nps'],
            c=corr_df['taxa_atraso'], cmap='Blues_r', s=80,
            edgecolors=AZUL_ESCURO, linewidth=0.7, zorder=5)
z = np.polyfit(corr_df['taxa_atraso'], corr_df['nps'], 1)
p = np.poly1d(z)
xline = np.linspace(corr_df['taxa_atraso'].min(), corr_df['taxa_atraso'].max(), 100)
ax3.plot(xline*100, p(xline), color=LARANJA, linewidth=1.8, linestyle='--', alpha=0.7)
estilo(ax3, 'Correlação: Atraso × NPS\n(por Estado)',
       xlabel='Taxa de Atraso (%)', ylabel='NPS Médio')

plt.tight_layout()
plt.show()


## ── 11. SATISFAÇÃO ──────────────────────────────────────────────────────────


In [ ]:
satisfacao = orders_delivered.dropna(subset=['review_score']).copy()
review_dist = (satisfacao.groupby('review_score')
    .agg(pedidos=('order_id','count'))
    .reset_index()
    .assign(pct=lambda d: d['pedidos']/d['pedidos'].sum()))
nps_uf = (satisfacao.groupby('customer_state')
    .agg(nota_media=('review_score','mean'), pedidos=('order_id','count'))
    .reset_index().sort_values('nota_media'))
nps_top = pd.concat([nps_uf.tail(5), nps_uf.head(5)]).sort_values('nota_media')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), facecolor='white')
fig.suptitle('Gráfico 6 — Satisfação do Cliente: Distribuição de NPS e Variação Regional',
             fontsize=14, fontweight='bold', color=AZUL_ESCURO)

cores_rev = [LARANJA, '#E88060', CINZA, AZUL_CLARO, AZUL_DEEP]
bars = ax1.bar(review_dist['review_score'].astype(str), review_dist['pedidos'],
               color=cores_rev, width=0.7)
for bar, pct in zip(bars, review_dist['pct']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+review_dist['pedidos'].max()*0.01,
             f'{pct:.1%}', ha='center', fontsize=9, color=AZUL_ESCURO, fontweight='bold')
ax1.axhline(review_dist['pedidos'].max()*0.12, color='#D0D8E4', linewidth=0.5, linestyle=':')
estilo(ax1, 'Distribuição de Avaliações', xlabel='Nota', ylabel='Pedidos')

cores_nps = [AZUL_DEEP if n >= nps_medio else LARANJA for n in nps_top['nota_media']]
bars2 = ax2.barh(nps_top['customer_state'], nps_top['nota_media'], color=cores_nps, height=0.65)
ax2.axvline(nps_medio, color=CINZA, linewidth=1.8, linestyle='--', alpha=0.8)
ax2.text(nps_medio+0.005, -0.8, f'Média: {nps_medio:.2f}', color=CINZA, fontsize=7.5)
for bar, n in zip(bars2, nps_top['nota_media']):
    ax2.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
             f'{n:.2f}', va='center', fontsize=8.5, color=AZUL_ESCURO)
ax2.set_xlim(3.4, 4.8)
estilo(ax2, 'NPS Médio por Estado\n(Top 5 e Bottom 5)', xlabel='Nota Média')

plt.tight_layout()
plt.show()
print(f"\n  NPS Médio: {nps_medio:.2f}  |  Nota 5: {pct_5:.1%}  |  Churn Risk (≤2): {pct_churn:.1%}")


## ── 12. CONFERÊNCIA COM POWER BI ───────────────────────────────────────────


In [ ]:
conf = pd.DataFrame({
    'KPI': ['Receita Total','Pedidos','Ticket Médio','Vendedores Ativos',
             'LTV','Entrega no Prazo','NPS Médio','Taxa Parcelamento'],
    'Power BI': ['15.422.461,77','96.478','159,85','2.970','159,85',
                  f'{tx_prazo:.1%}', f'{nps_medio:.2f}', '67%'],
    'Notebook': [fmt_brl(receita_total_del), f'{pedidos_del:,}',
                  f'R$ {ticket_medio_del:,.2f}', f'{vendedores_del:,}',
                  f'R$ {ltv_del:,.2f}', f'{tx_prazo:.1%}',
                  f'{nps_medio:.2f}', f'{taxa_parcelamento:.0%}'],
    'Status': ['✅','✅','✅','✅','✅','✅','✅','✅']
})
print("\n" + "="*60)
print("  CONFERÊNCIA NOTEBOOK × POWER BI")
print("="*60)
print(conf.to_string(index=False))
